In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
scaler = joblib.load("../models/scaler.joblib")
final_model = joblib.load("../models/final_model.joblib")
threshold = joblib.load("../models/threshold.joblib")
diagnosis_models = joblib.load("../models/diagnosis_models.joblib")

print("Threshold:", threshold)
print("Diagnosis models:", list(diagnosis_models.keys()))

Threshold: 0.6000000000000002
Diagnosis models: ['TWF', 'HDF', 'PWF', 'OSF']


In [3]:
feature_cols = list(pd.read_csv("../data/processed/X_train.csv", nrows=1).columns)
feature_cols

['Type',
 'air_temp',
 'process_temp',
 'rpm',
 'torque',
 'tool_wear',
 'temp_diff',
 'power',
 'wear_torque']

In [4]:
def build_features(machine_type, air_temp, process_temp, rpm, torque, tool_wear):
    type_map = {"L": 0, "M": 1, "H": 2}

    row = {
        "Type": type_map[machine_type],
        "air_temp": air_temp,
        "process_temp": process_temp,
        "rpm": rpm,
        "torque": torque,
        "tool_wear": tool_wear,
        "temp_diff": process_temp - air_temp,
        "power": torque * rpm * 2 * np.pi / 60,
        "wear_torque": tool_wear * torque,
    }

    return pd.DataFrame([row])[feature_cols]

In [5]:
def get_risk_level(prob):
    if prob >= 0.60:
        return "High"
    elif prob >= 0.30:
        return "Medium"
    else:
        return "Low"

In [6]:
def get_recommendations(machine_type, air_temp, process_temp, rpm, torque, tool_wear):
    advice = []

    temp_diff = process_temp - air_temp
    power = torque * rpm * 2 * np.pi / 60
    wear_torque = tool_wear * torque
    overstrain_limit = {"L": 11000, "M": 12000, "H": 13000}[machine_type]

    if tool_wear >= 200:
        advice.append("Tool wear is high: replace or inspect the tool.")

    if temp_diff < 8.6 and rpm < 1380:
        advice.append("Low temperature difference at low speed: check the cooling system.")

    if power < 3500:
        advice.append("Power is too low: check for slipping or an unstable load.")
    elif power > 9000:
        advice.append("Power is too high: reduce the load or speed.")

    if wear_torque > overstrain_limit:
        advice.append("Overstrain risk: reduce torque on this worn tool.")

    if not advice:
        advice.append("All readings are in the normal range. Continue regular monitoring.")

    return advice

In [7]:
def predict_machine(machine_type, air_temp, process_temp, rpm, torque, tool_wear):
    # 1. features and scaling
    features = build_features(machine_type, air_temp, process_temp, rpm, torque, tool_wear)
    scaled = pd.DataFrame(scaler.transform(features), columns=feature_cols)

    # 2. failure probability and risk level
    prob = final_model.predict_proba(scaled)[0, 1]
    risk = get_risk_level(prob)

    # 3. likely failure type (probability for each type)
    type_probs = {
        t: diagnosis_models[t].predict_proba(scaled)[0, 1]
        for t in diagnosis_models
    }
    likely_cause = max(type_probs, key=type_probs.get)

    # 4. advice
    advice = get_recommendations(machine_type, air_temp, process_temp, rpm, torque, tool_wear)

    return {
        "failure_probability": round(float(prob), 3),
        "failure_alert": bool(prob >= threshold),
        "risk_level": risk,
        "likely_cause": likely_cause,
        "cause_probabilities": {k: round(float(v), 3) for k, v in type_probs.items()},
        "advice": advice,
    }

Test

In [8]:
predict_machine("L", 298, 308.6, 1500, 40, 50)

{'failure_probability': 0.0,
 'failure_alert': False,
 'risk_level': 'Low',
 'likely_cause': 'PWF',
 'cause_probabilities': {'TWF': 0.0, 'HDF': 0.0, 'PWF': 0.0, 'OSF': 0.0},
 'advice': ['All readings are in the normal range. Continue regular monitoring.']}

In [9]:
predict_machine("L", 298, 308.6, 1400, 60, 220)

{'failure_probability': 1.0,
 'failure_alert': True,
 'risk_level': 'High',
 'likely_cause': 'OSF',
 'cause_probabilities': {'TWF': 0.061, 'HDF': 0.0, 'PWF': 0.0, 'OSF': 1.0},
 'advice': ['Tool wear is high: replace or inspect the tool.',
  'Overstrain risk: reduce torque on this worn tool.']}

In [10]:
predict_machine("M", 302, 310, 1300, 45, 80)

{'failure_probability': 1.0,
 'failure_alert': True,
 'risk_level': 'High',
 'likely_cause': 'HDF',
 'cause_probabilities': {'TWF': 0.0, 'HDF': 1.0, 'PWF': 0.001, 'OSF': 0.0},
 'advice': ['Low temperature difference at low speed: check the cooling system.']}

In [11]:
predict_machine("H", 298, 308.6, 1300, 70, 100)

{'failure_probability': 1.0,
 'failure_alert': True,
 'risk_level': 'High',
 'likely_cause': 'PWF',
 'cause_probabilities': {'TWF': 0.0, 'HDF': 0.006, 'PWF': 1.0, 'OSF': 0.0},
 'advice': ['Power is too high: reduce the load or speed.']}

In [12]:
raw = pd.read_csv("../data/ai4i2020.csv")
raw = raw.rename(columns={
    "Air temperature [K]": "air_temp",
    "Process temperature [K]": "process_temp",
    "Rotational speed [rpm]": "rpm",
    "Torque [Nm]": "torque",
    "Tool wear [min]": "tool_wear",
    "Machine failure": "failure"
})

failed = raw[raw["failure"] == 1].head(5)

for _, r in failed.iterrows():
    result = predict_machine(r["Type"], r["air_temp"], r["process_temp"],
                             r["rpm"], r["torque"], r["tool_wear"])
    print(r["Type"], "| prob:", result["failure_probability"],
          "| risk:", result["risk_level"], "| cause:", result["likely_cause"])

L | prob: 0.999 | risk: High | cause: PWF
L | prob: 1.0 | risk: High | cause: OSF
L | prob: 0.988 | risk: High | cause: TWF
L | prob: 1.0 | risk: High | cause: OSF
L | prob: 0.998 | risk: High | cause: OSF
